In [0]:

%pip install geopy requests sentence-transformers trafilatura

In [0]:
%restart_python

In [0]:
dbutils.widgets.text("nws_api_base_url", "https://api.weather.gov", "NWS API base URL")
dbutils.widgets.text("alert_table_name", "weather_alert_documents", "WEATHER ALERT TABLE")
dbutils.widgets.text("embedding_model", "sentence-transformers/all-MiniLM-L6-v2", "Embedding model")
dbutils.widgets.text("embeddings_table_name", "weather_alert_embeddings", "Alert Embeddings")
dbutils.widgets.text("chunk_size", "800", "Alert content chunk size (chars)")
dbutils.widgets.text("chunk_overlap", "100", "Alert content chunk overlap (chars)")


NWS_API_BASE_URL = dbutils.widgets.get("nws_api_base_url")
ALERT_TABLE_NAME = dbutils.widgets.get("alert_table_name")
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model")
EMBEDDINGS_TABLE_NAME = dbutils.widgets.get("embeddings_table_name")
CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def get_lakebase_url() -> str:
    secret = w.secrets.get_secret(scope="database", key="lakebase-url")
    return base64.b64decode(secret.value).decode("utf-8")


lakebase_url = get_lakebase_url()
parsed = urlparse(lakebase_url)

# Extract connection details directly from the secret URL
db_host = parsed.hostname
db_port = parsed.port or 5432
db_name = parsed.path.lstrip('/')
db_user = parsed.username
db_password = parsed.password

print(f"Connection details:")
print(f"  Host: {db_host}:{db_port}")
print(f"  Database: {db_name}")
print(f"  User: {db_user}")
print(f"  Using raw credentials from secret (no OAuth)")

In [0]:
import logging
from weather_client import NWSClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Initialize NWS client
client = NWSClient(base_url=NWS_API_BASE_URL)

# Define cities to fetch alerts for
location_list = [
    "Boston, MA", 
    "Austin, TX", 
    "New York, NY", 
    "Denver, CO", 
    "San Francisco, CA"
]

print(f"Fetching alerts for {len(location_list)} cities...\n")

try:
    # Fetch and normalize alerts using the client
    all_alerts = client.fetch_alerts_for_cities(location_list)
    
    print(f"\n✅ Successfully collected {len(all_alerts)} alerts")
    print(f"Sample alert: {all_alerts[0] if all_alerts else 'None'}")
    
except Exception as e:
    print(f"❌ Error fetching alerts: {e}")
    raise
finally:
    client.close()
        







In [0]:
import lakebase
from embedding_utils import batch_insert_alerts_to_lakebase

# Insert alerts using the utility function
try:
    with lakebase.get_connection() as conn:
        inserted_count = batch_insert_alerts_to_lakebase(
            alerts=all_alerts,
            conn=conn,
            alert_table=ALERT_TABLE_NAME
        )
        print(f"\n✅ Inserted {inserted_count} new alerts into {ALERT_TABLE_NAME}")
except Exception as e:
    print(f"❌ Error inserting alerts: {e}")
    raise

In [0]:
import lakebase
from chunking_utils import load_all_alerts_from_db

# Load alerts from database
try:
    with lakebase.get_connection() as conn:
        alerts_df = load_all_alerts_from_db(
            conn=conn,
            alert_table=ALERT_TABLE_NAME
        )
    
    print(f"\nLoaded {len(alerts_df)} alerts for embedding")
    display(alerts_df.head(5))
    
except Exception as e:
    print(f"❌ Error loading alerts: {e}")
    raise

In [0]:
from chunking_utils import chunk_alerts_dataframe

# Chunk the alerts using the utility function
try:
    chunks_df = chunk_alerts_dataframe(
        alerts_df=alerts_df,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        text_column="embedding_text"
    )
    
    print(f"\nGenerated {len(chunks_df)} chunks from {len(alerts_df)} alerts")
    display(chunks_df.head(5))
    
except Exception as e:
    print(f"❌ Error chunking alerts: {e}")
    raise

In [0]:
from embedding_utils import EmbeddingGenerator

# Initialize embedding generator
embedding_gen = EmbeddingGenerator(model_name=EMBEDDING_MODEL_NAME)

try:
    # Generate embeddings for all chunks
    chunk_embeddings_df = embedding_gen.embed_chunks(
        chunks_df=chunks_df,
        batch_size=32
    )
    
    print(f"\n✅ Computed {len(chunk_embeddings_df)} embeddings using {EMBEDDING_MODEL_NAME}")
    print(f"Sample embedding dimensions: {len(chunk_embeddings_df.iloc[0]['embedding'])}")
    
except Exception as e:
    print(f"❌ Error computing embeddings: {e}")
    raise

In [0]:
import lakebase
from embedding_utils import insert_embeddings_to_lakebase

# Insert embeddings using the utility function
try:
    with lakebase.get_connection() as conn:
        inserted_count = insert_embeddings_to_lakebase(
            embeddings_df=chunk_embeddings_df,
            conn=conn,
            embeddings_table=EMBEDDINGS_TABLE_NAME
        )
    
    print(f"\n✅ Successfully inserted {inserted_count} embeddings into {EMBEDDINGS_TABLE_NAME}")
    print(f"\nPipeline complete! {len(all_alerts)} alerts → {len(chunks_df)} chunks → {inserted_count} embeddings")
    
except Exception as e:
    print(f"❌ Error inserting embeddings: {e}")
    raise